In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/fasgadhsxnzmjj/mp4-dataset/SON TUNG M-TP - MAKING MY WAY - OFFICIAL VISUALIZER - YouTube.mp4
/kaggle/input/datasets/fasgadhsxnzmjj/mp4-dataset/treemp3.mp3
/kaggle/input/datasets/fasgadhsxnzmjj/mp4-dataset/treemp4.mp4


In [3]:
!pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 53.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 47.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.5 MB/s eta 0:00:00:00:0100:01


In [5]:
# Chạy trực tiếp trong cell của Kaggle
!ffmpeg -i "/kaggle/input/datasets/fasgadhsxnzmjj/mp4-dataset/SON TUNG M-TP - MAKING MY WAY - OFFICIAL VISUALIZER - YouTube.mp4" -vn -acodec libmp3lame -q:a 2 /kaggle/working/output.mp3

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [10]:
from faster_whisper import WhisperModel
import time

def speech_to_text_pipeline(mp3_path, model_size="base"):
    # 1. Khởi tạo Model 
    # model_size có thể là: 'tiny', 'base', 'small', 'medium', 'large-v3'
    # device="cuda" nếu có GPU, hoặc "cpu" nếu chạy trên máy thường
    print(f"--- Đang tải model {model_size} ---")
    model = WhisperModel(model_size, device="cuda", compute_type="float16")

    # 2. Xử lý file MP3
    # beam_size: số lượng luồng tìm kiếm (càng cao càng chính xác nhưng chậm hơn)
    # language="vi": chỉ định tiếng Việt để tăng độ chính xác (hoặc để None để tự nhận diện)
    print(f"--- Đang xử lý file: {mp3_path} ---")
    start_time = time.time()
    
    segments, info = model.transcribe(mp3_path, beam_size=5, language="en")

    print(f"Ngôn ngữ được nhận diện: {info.language} (Độ tin cậy: {info.language_probability:.2f})")

    # 3. Gom kết quả và in ra
    full_text = []
    for segment in segments:
        text_line = f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}"
        print(text_line)
        full_text.append(segment.text)

    end_time = time.time()
    print(f"\n--- Hoàn thành trong {end_time - start_time:.2f} giây ---")
    
    return " ".join(full_text)

# Chạy pipeline
if __name__ == "__main__":
    mp3_file = "/kaggle/working/output.mp3" 
    result = speech_to_text_pipeline(mp3_file, model_size="medium")

--- Đang tải model medium ---
--- Đang xử lý file: /kaggle/working/output.mp3 ---
Ngôn ngữ được nhận diện: en (Độ tin cậy: 1.00)
[0.00s -> 27.00s]  We're looking bright tonight High in the sky, let's try to keep you alive
[27.00s -> 32.00s]  Alright, just stay away from me Let me down easily
[32.00s -> 37.00s]  You're too light, you're too light Don't you pray, don't you pray
[37.00s -> 42.00s]  Seeking for my embrace Wanting my touch, begging me to stay
[42.00s -> 47.00s]  Every night and day Calling my name, shout it till you break
[47.00s -> 51.00s]  You took my heart Held it and ripped it apart
[51.00s -> 56.00s]  Make me your prisoner Get me going right under
[56.00s -> 60.00s]  Feeling that you'll find Find me again
[60.00s -> 65.00s]  Nothing to explain Talking my own land
[65.00s -> 70.00s]  God's helping me out Get over my doubt
[70.00s -> 76.00s]  Thought I can't live without Fuckin' you right now
[76.00s -> 96.00s]  Making my way, making my way
[96.00s -> 101.00s]  I wanna k

In [11]:
print(result)

 We're looking bright tonight High in the sky, let's try to keep you alive  Alright, just stay away from me Let me down easily  You're too light, you're too light Don't you pray, don't you pray  Seeking for my embrace Wanting my touch, begging me to stay  Every night and day Calling my name, shout it till you break  You took my heart Held it and ripped it apart  Make me your prisoner Get me going right under  Feeling that you'll find Find me again  Nothing to explain Talking my own land  God's helping me out Get over my doubt  Thought I can't live without Fuckin' you right now  Making my way, making my way  I wanna know the cure For all that time you made me injure  All the words you said I'll keep your prayer well and behave  I reckon it's time Pedding my luck on dance  Take a new leap Getting far away from your cheating  Seeking for my embrace Wanting my touch, begging me to stay  Every night and day Calling my name, shout it till you break  You took my heart Held it and ripped it ap

In [24]:
!pip install transformers==4.37.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 85.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.1 MB/s eta 0:00:0000:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

In [34]:
!pip show transformers

Name: transformers
Version: 4.37.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: kaggle-environments, peft, sentence-transformers


In [39]:
# Transformers không tương thích với Phowhisper chạy thế nào cũng không được đã cài bản transformers cũ nhưng vẫn lỗi
from transformers import pipeline
transcriber = pipeline("automatic-speech-recognition", model="vinai/PhoWhisper-small")
output = transcriber("/kaggle/working/treemp3_chunk_0.mp3")['text']

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 657, in hf_raise_for_status
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/vinai/PhoWhisper-small/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion

KeyError: 'num_frames'

In [48]:
import torch
import librosa
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

model_id = "vinai/PhoWhisper-small"

# Load processor + model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id)

# fix warning (optional)
model.config.tie_word_embeddings = False

# load audio
audio, sr = librosa.load("/kaggle/working/treemp3_chunk_0.mp3", sr=16000)

# convert -> input features
inputs = processor(
    audio,
    sampling_rate=sr,
    return_tensors="pt"
)

# generate
with torch.no_grad():
    generated_ids = model.generate(inputs["input_features"])

# decode
text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

print(text)

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 657, in hf_raise_for_status
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/vinai/PhoWhisper-small/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion

xin chào và rất vui với tất cả các bạn bản năng tổ tiên khiến chúng ta phấn khích trước những thứ mới mẻ kỳ lạ hoặc bất quỳ tắc đó là lý do mà.


In [ ]:
import torch 
import librosa 
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq 
import subprocess 
from pydub import AudioSegment 
import os 

model_id = "vinai/PhoWhisper-small" # Load processor + model 

processor = AutoProcessor.from_pretrained(model_id) 
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id) 

# fix warning (optional) 
model.config.tie_word_embeddings = False

In [5]:
class Mp4_to_text:
    def __init__(self, language, input_file_path_mp4, time_chunk_mp3, folder_path_chunk, output_file_path_mp3, output_file_path_txt,gpu): 
        # ngôn ngữ của video đầu vào hiện tại chỉ lấy tiếng việt. Trong tương lai sẽ update và thêm các model để xử lý video tiếng nước ngoài
        self.language = language
        # đường dẫn của video mp4 đầu vào
        self.ip_path = input_file_path_mp4
        # độ dài mỗi chunk để model trích xuất khuyến nghị không nên sửa để tránh model bị quá tham số -> lỗi
        self.chunk_length_ms = time_chunk_mp3
        # đường dẫn của folder tổng lưu trữ chunk
        self.base_folder =  folder_path_chunk
        # đường dẫn của file audio mp3 sau khi được chuyển từ video mp4
        self.op_file_mp3 = output_file_path_mp3
        # đường dẫn kết quả trả về file txt
        self.op_file_txt = output_file_path_txt
        # nếu máy có gpu thì là True không thì là False tối ưu cho tốc độ nếu có gpu
        self.gpu = "True"
        

        # ví dụ:
        # self.language = "vi"
        # self.ip_path = "/kaggle/input/datasets/fasgadhsxnzmjj/mp4-dataset/treemp4.mp4"
        # self.chunk_length_ms = 30 * 1000 
        # self.base_folder = "/kaggle/working/filemp3_chunk"
        # self.op_file_mp3 = "/kaggle/working/output2.mp3"
        # self.gpu = "True"

    def split_audio_into_chunks(self, file_path):
        folder = os.path.join(
            self.base_folder,
            os.path.splitext(os.path.basename(file_path))[0]
        )

        os.makedirs(folder, exist_ok=True)

        audio = AudioSegment.from_file(file_path)
        duration = len(audio)

        for i, start in enumerate(range(0, duration, self.chunk_length_ms)):
            chunk = audio[start:start + self.chunk_length_ms]
            chunk_path = os.path.join(folder, f"chunk_{i}.mp3")
            chunk.export(chunk_path, format="mp3")

        return folder

    def mp4_to_mp3(self):
        command = [
            "ffmpeg", "-i", self.ip_path,
            "-vn",
            "-acodec", "libmp3lame",
            "-ab", "192k",
            self.op_file_mp3
        ]
        subprocess.run(command, check=True)

    def mp3_to_text_phoWhisper(self, file_mp3):
        audio, sr = librosa.load(file_mp3, sr=16000)

        inputs = processor(
            audio,
            sampling_rate=sr,
            return_tensors="pt"
        )

        with torch.no_grad():
            ids = model.generate(inputs["input_features"])

        return processor.batch_decode(ids, skip_special_tokens=True)[0]

    def batch_mp3_to_text(self, file_list):
        # dùng nếu máy có GPU
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        
        audios = []
    
        # Load toàn bộ audio
        for file_mp3 in file_list:
            audio, sr = librosa.load(file_mp3, sr=16000)
            audios.append(audio)
    
        # Convert batch
        inputs = processor(
            audios,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True   # QUAN TRỌNG
        )
    
        # Đưa lên GPU
        inputs = {k: v.to(device) for k, v in inputs.items()}
    
        # Inference 1 lần
        with torch.no_grad():
            generated_ids = model.generate(
                inputs["input_features"],
                max_new_tokens=256,
                do_sample=False
            )
    
        # Decode batch
        texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
    
        return texts

    def save_to_txt(self, text, output_path="output.txt"):
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)
    
    def op_vi(self):
        result = []

        self.mp4_to_mp3()
        folder = self.split_audio_into_chunks(self.op_file_mp3)

        files = sorted(
            [f for f in os.listdir(folder) if f.endswith(".mp3")],
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )

        if self.gpu == "False":
            for f in files:
                path = os.path.join(folder, f)
                text = self.mp3_to_text_phoWhisper(path)
                result.append(text)

        if self.gpu == "True":
            file_paths = [os.path.join(folder, f) for f in files]
            # chia batch (rất quan trọng nếu nhiều file)
            batch_size = 4
        
            for i in range(0, len(file_paths), batch_size):
                batch_files = file_paths[i:i + batch_size]
                texts = self.batch_mp3_to_text(batch_files)
                result.extend(texts)
        
        # return result
        full_text = " ".join(result)
        self.save_to_txt(full_text, self.op_file_txt)
        return full_text


# ngôn ngữ của video đầu vào hiện tại chỉ lấy tiếng việt. Trong tương lai sẽ update và thêm các model để xử lý video tiếng nước ngoài
language = "vi"
# đường dẫn của video mp4 đầu vào
input_file_path_mp4 = "/kaggle/input/datasets/fasgadhsxnzmjj/mp4-dataset/treemp4.mp4"
# độ dài mỗi chunk để model trích xuất khuyến nghị không nên sửa để tránh model bị quá tham số -> lỗi
time_chunk_mp3 = 30 * 1000
# đường dẫn của folder tổng lưu trữ chunk
folder_path_chunk = "/kaggle/working/filemp3_chunk"
# đường dẫn của file audio mp3 sau khi được chuyển từ video mp4
output_file_path_mp3 = "/kaggle/working/output3.mp3"
# đường dẫn kết quả trả về file txt
output_file_path_txt = "/kaggle/working/output.txt"
# nếu máy có gpu thì là True không thì là False tối ưu cho tốc độ nếu có gpu
gpu = "True"

test = Mp4_to_text(language, input_file_path_mp4, time_chunk_mp3, folder_path_chunk, output_file_path_mp3, output_file_path_txt, gpu)
print(test.op_vi())

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

xin chào và rất vui với tất cả các bạn bản năng tổ tiên khiến chúng ta phấn khích trước những thứ mới mẻ kỳ lạ hoặc bất quỳ tắc đó là lý do mà những bộ phim lên bối cảnh thế giới ngoài hành tinh thì luôn có một sức hút khó giải thích nhưng bạn biết gì không nếu người ngoài hành tinh ghé thẳng trái đất họ cũng sẽ cảm thấy rằng thế giới của chúng ta kỳ lạ thậm chí không thể tìm được và thứ tạo ra cảm giác ồ hoáu đó không phải là con người hay bất kỳ. kỳ loài động vật nào khác nó là vật thứ khác một thứ mà nếu như chỉ nghe kể giống như thuộc về thế giới thần thoại hơn là phiên bản đời thực chúng khổng lồ sống lâu và có cấu trúc kỳ dị chúng không tồn tại trong phần loại sinh học và thậm chí sở hữu công nghệ mà còn người khao khát phản trọng lực chúng ta đang nói đến cây ba trước khi mà bạn phẫn nộ cho chúng ta xin mười phút để giải thích nhé. đầu tiên cây không thực sự tồn tại trong phần loại sinh học đừng để từ cây đánh lừa bạn có hàng vạn loài trên thế giới bắt đầu bằng chữ cây hoặc được